# 06 — GSV Evaluation: Final Results Report

Final benchmark of skyline-only geolocation on **68 multi-crop GSV panoramas**
(Khumbu region) against the 1.34M-viewpoint horizon database.

**Scorers compared**
- `baseline` — value + gradient feature NCC (production default)
- `bp28` / `bp316` — DoG bandpass NCC, σ 2→8 and 3→16
- `rrf` — reciprocal-rank fusion of all three

**Confidence gate:** report a match only when all three scorers' top-1
predictions agree (`rrf_votes == 3`); abstain otherwise.

Data source: `final/results/gsv_improve_eval_results.json`
(produced by `scripts/gsv_improve_eval.py`).

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'street_view').exists() or (p / 'final').exists())
CANDIDATES = [ROOT / 'final' / 'results' / 'gsv_improve_eval_results.json',
              ROOT / 'data' / 'street_view' / 'gsv_improve_eval_results.json']
JSON_PATH = next(c for c in CANDIDATES if c.exists())
results = json.loads(JSON_PATH.read_text())['results']
print(f'{len(results)} panoramas loaded from {JSON_PATH}')

## 1. Method comparison

In [ ]:
def stats(es):
    es = np.asarray(es, float); es = es[np.isfinite(es)]
    return dict(N=len(es), median_km=np.median(es)/1e3,
                p100=np.mean(es<100), p1k=np.mean(es<1000), p10k=np.mean(es<10000))

SC = ['baseline', 'bp28', 'bp316', 'rrf']
hdr = f"{'scorer':<10} {'N':>3} {'median':>8} {'<100m':>7} {'<1km':>7} {'<10km':>7}"
print(hdr); print('-'*len(hdr))
for s in SC:
    r = stats([x[f'err_{s}'] for x in results])
    print(f"{s:<10} {r['N']:>3} {r['median_km']:>6.1f}km {r['p100']:>6.1%} {r['p1k']:>6.1%} {r['p10k']:>6.1%}")
orc = stats([min(x[f'err_{s}'] for s in SC) for x in results])
print(f"{'oracle':<10} {orc['N']:>3} {orc['median_km']:>6.1f}km {orc['p100']:>6.1%} {orc['p1k']:>6.1%} {orc['p10k']:>6.1%}")

wide = [x for x in results if x['coverage_deg'] >= 200]
print('\n--- Wide-FOV subset (>=200 deg coverage) ---')
for s in SC + ['oracle']:
    es = ([min(x[f'err_{c}'] for c in SC) for x in wide] if s=='oracle'
          else [x[f'err_{s}'] for x in wide])
    r = stats(es)
    print(f"{s:<10} {r['N']:>3} {r['median_km']:>6.1f}km {r['p100']:>6.1%} {r['p1k']:>6.1%} {r['p10k']:>6.1%}")

## 2. Confidence gating — precision when we claim a match

In [ ]:
def gate(name, subset):
    errs = np.array([x['err_rrf'] for x in subset], float)
    errs = errs[np.isfinite(errs)]
    print(f"{name:<28} N={len(errs):>2}  <100m={np.mean(errs<100):>6.1%}  "
          f"<1km={np.mean(errs<1000):>6.1%}  median={np.median(errs):>5.0f}m")

gate('all panos (no gate)', results)
gate('consensus only (votes>=3)', [x for x in results if x['rrf_votes'] >= 3])
gate('wide-FOV only (>=200deg)', wide)
gate('wide-FOV + consensus', [x for x in wide if x['rrf_votes'] >= 3])
rej = [x for x in results if not (x['coverage_deg']>=200 and x['rrf_votes']>=3)]
gate('REJECTED by gate (would be)', rej)

## 3. Consensus-accepted matches

In [ ]:
acc = sorted((x for x in results if x['rrf_votes'] >= 3), key=lambda x: x['err_rrf'])
print(f"{'pano':<20} {'FOV':>4}  {'base':>8} {'bp28':>8} {'bp316':>8} {'RRF':>9}")
for x in acc:
    hit = 'HIT ' if x['err_rrf'] < 1000 else 'miss'
    print(f"[{hit}] {x['pano_id'][:16]:<16} {x['coverage_deg']:>4.0f}  "
          f"{x['err_baseline']/1e3:>6.1f}km {x['err_bp28']/1e3:>6.1f}km "
          f"{x['err_bp316']/1e3:>6.1f}km {x['err_rrf']:>7.0f}m")

## 4. Figures

Regenerate with `python scripts/make_report_figures.py`; PNGs live in
`final/figures/`.

| Figure | Content |
|---|---|
| `fig1_error_cdf` | error CDFs, all panos + wide-FOV subset |
| `fig2_confidence_gates` | precision by confidence gate |
| `fig3_baseline_vs_rrf` | per-pano scatter, baseline vs fusion |
| `fig4_fov_vs_error` | fused-horizon coverage vs achievable accuracy |

In [ ]:
from IPython.display import Image, display
figdir = ROOT / 'final' / 'figures'
if figdir.exists():
    for f in ['fig1_error_cdf', 'fig2_confidence_gates']:
        p = figdir / f'{f}.png'
        if p.exists():
            display(Image(filename=str(p), width=800))